In [96]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [97]:
import gym
import numpy as np
import random

In [98]:
env = gym.make('CartPole-v1')

In [99]:
class VanillaFeatureEncoder:
    def __init__(self, env):
        self.env = env
        
    def encode(self, state):
        return state
    
    @property
    def size(self):
        return self.env.observation_space.shape[0]

In [100]:
class QLearning_LVFA:
    def __init__(self, env, feature_encoder_cls, alpha=0.005, 
                 alpha_decay=0.9999, gamma=0.9999, epsilon=1., epsilon_decay=0.9999):
        self.env = env
        self.feature_encoder = feature_encoder_cls(env)
        self.shape = (self.env.action_space.n, self.feature_encoder.size)
        # self.weights = np.zeros(self.shape)
        self.weights = np.random.random(self.shape)
        self.alpha = alpha
        self.alpha_decay = alpha_decay
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay


    # Q(2x1) = W(2x4) * X(4x1) 
    def Q(self, feats):
        #take all te dimensions, put it on the left, so i will have a column vector
        feats = feats.reshape(-1,1)
        return self.weights @ feats
    
        
    #Q-learning
    def policy(self, state):
        state_feats = self.feature_encoder.encode(state)
        return np.argmax(Q(state_feats))

    #SARSA
    def epsilon_greedy(self, state, epsilon=None):

        if epsilon is None: epsilon = self.epsilon

        if random.random() < epsilon:
            return self.env.action_space.sample() 
        else:
            return self.policy(state)

        return 0
    
    def update_transition(self, s, action, s_prime, reward, done):
        s_feats = self.feature_encoder.encode(s)
        s_prime_feats = self.feature_encoder.encode(s_prime)
        action_prime = self.epsilon_greedy(s_prime)
        td_error = reward
        
        
        delta_w = np.zeros(self.feature_encoder.size)

        q_a_prime = self.Q(s_prime_feats)[action_prime]

        delta_w = (reward + self.gamma*(q_a_prime - (self.Q(s_feats))[action]) @ np.diff((self.Q(s_feats))[action])
        
        self.weights[action] += self.alpha*delta_w
        

    def update_alpha_epsilon(self):
        self.epsilon = max(0.2, self.epsilon*self.epsilon_decay)
        self.alpha = self.alpha*self.alpha_decay
       
        
    def train(self, n_episodes=200, max_steps_per_episode=200):
        for episode in range(n_episodes):
            done = False
            s, _ = env.reset()
            for i in range(max_steps_per_episode):

                #if all(s == env.end_state):
                 #   continue  # if we reach the termination condition, we cannot perform any action
                
                action = self.epsilon_greedy(s)
                s_prime, reward, done, _, _ = self.env.step(action)
                self.update_transition(s, action, s_prime, reward, done)
                
                s = s_prime
                
                if done: break
                
            self.update_alpha_epsilon()

            if episode % 20 == 0:
                print(episode, self.evaluate(), self.epsilon, self.alpha)

                
    def evaluate(self, env=None, n_episodes=10, max_steps_per_episode=200):
        if env is None:
            env = self.env
            
        rewards = []
        for episode in range(n_episodes):
            total_reward = 0
            done = False
            s, _ = env.reset()
            for i in range(max_steps_per_episode):
                action = 0

                action = epsilon_greedy(s)
                
                s_prime, reward, done, _, _ = env.step(action)
                
                total_reward += reward
                s = s_prime
                if done: break
            
            rewards.append(total_reward)
            
        return np.mean(rewards)


SyntaxError: '(' was never closed (4097257069.py, line 51)

In [ ]:
agent = QLearning_LVFA(env, VanillaFeatureEncoder)
agent.train()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
agent.evaluate()

9.0

In [ ]:
agent.weights

array([[0.9813034 , 0.52524298, 0.55870543, 0.00467152],
       [0.90703909, 0.94690915, 0.01408159, 0.29391681]])